In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


# MLFlow Initialization

In [2]:
%pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 76.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import dagshub
dagshub.init(repo_owner='izere23', repo_name='Assignment-2-IEEE-CIS-Fraud-Detection', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a099ccb7-aed5-4de2-ae7d-3e798f326b72&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7ad4b26bc7256da4b952c950b5be0856869715da09a7265b47413848ff93bcb9




Output()

Accessing as izere23

Initialized MLflow to track repo "izere23/Assignment-2-IEEE-CIS-Fraud-Detection"

Repository izere23/Assignment-2-IEEE-CIS-Fraud-Detection initialized!

In [ ]:
import mlflow

# Data Loading

In [5]:
BASE_PATH = '/kaggle/input/competitions/ieee-fraud-detection/'
train_transaction = pd.read_csv(BASE_PATH + "train_transaction.csv")
train_identity = pd.read_csv(BASE_PATH + "train_identity.csv")

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)
print(train["isFraud"].value_counts(normalize=True))

(590540, 434)
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [6]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=["isFraud"])
y = train["isFraud"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_val.shape)
print(y_train.mean(), y_val.mean())

(472432, 433) (118108, 433)
0.03498916246147594 0.0349933958749619


# Cleaning

## High Missing Columns

In [19]:
class HighMissingDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold
        self.cols_to_drop_ = None

    def fit(self, X, y=None):
        self.cols_to_drop_ = X.columns[X.isna().mean() > self.threshold].tolist()
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

## High Dominance Columns (Unused)

In [23]:
from sklearn.base import BaseEstimator, TransformerMixin

class HighDominanceDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.995):
        self.threshold = threshold
        self.cols_to_drop_ = None

    def fit(self, X, y=None):
        dominance = X.apply(
            lambda col: col.value_counts(normalize=True, dropna=False).max()
        )

        self.cols_to_drop_ = dominance[
            dominance > self.threshold
        ].index.tolist()

        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

In [26]:
cleaning_pipeline = Pipeline([
    ("drop_high_missing", HighMissingDropper(threshold=0.98))

])

# Preprocessing

In [60]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from category_encoders.woe import WOEEncoder


class AutoPreprocessorWOE(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.preprocessor_ = None

    def fit(self, X, y=None):

        num_cols = X.select_dtypes(
            include=["int64", "float64"]
        ).columns.tolist()

        cat_cols = X.select_dtypes(
            include=["object", "category", "bool"]
        ).columns.tolist()

        num_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ])

        cat_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", WOEEncoder())
        ])

        self.preprocessor_ = ColumnTransformer([
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols)
        ])

        self.preprocessor_.fit(X, y)

        return self

    def transform(self, X):
        return self.preprocessor_.transform(X)

# Feature Engineering

## Log Adder (Not Used)

In [29]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np


class LogAmountAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "TransactionAmt" in X.columns:
            X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])

        return X

## UID (Only Deviation Used)

In [34]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np


class UIDAmountDeviationAdder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):

        X_temp = X.copy()

        X_temp["uid"] = (
            X_temp["card1"].astype(str) + "_" +
            X_temp["addr1"].astype(str)
        )

        self.uid_mean_ = (
            X_temp.groupby("uid")["TransactionAmt"]
            .mean()
            .to_dict()
        )

        return self

    def transform(self, X):

        X = X.copy()

        X["uid"] = (
            X["card1"].astype(str) + "_" +
            X["addr1"].astype(str)
        )

        X["uid_mean_amt"] = (
            X["uid"]
            .map(self.uid_mean_)
        )

        X["uid_amount_deviation"] = (
            X["TransactionAmt"] -
            X["uid_mean_amt"]
        )

        X.drop(columns=["uid"], inplace=True)

        return X

In [38]:
from sklearn.base import BaseEstimator, TransformerMixin

class UIDTransactionCountAdder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):

        X_temp = X.copy()

        X_temp["uid"] = (
            X_temp["card1"].astype(str) + "_" +
            X_temp["addr1"].astype(str)
        )

        self.uid_count_ = (
            X_temp["uid"]
            .value_counts()
            .to_dict()
        )

        return self

    def transform(self, X):

        X = X.copy()

        X["uid"] = (
            X["card1"].astype(str) + "_" +
            X["addr1"].astype(str)
        )

        X["uid_transaction_count"] = (
            X["uid"]
            .map(self.uid_count_)
        )

        X.drop(columns=["uid"], inplace=True)

        return X

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [41]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np


class UIDTimeDeltaAdder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):

        X_temp = X.copy()

        X_temp["uid"] = (
            X_temp["card1"].astype(str) + "_" +
            X_temp["addr1"].astype(str)
        )

        X_temp = X_temp.sort_values("TransactionDT")

        self.last_transaction_time_ = {}

        for _, row in X_temp.iterrows():
            self.last_transaction_time_[row["uid"]] = row["TransactionDT"]

        return self

    def transform(self, X):

        X = X.copy()

        X["uid"] = (
            X["card1"].astype(str) + "_" +
            X["addr1"].astype(str)
        )

        X = X.sort_values("TransactionDT")

        last_seen = {}

        deltas = []

        for _, row in X.iterrows():

            uid = row["uid"]
            current_time = row["TransactionDT"]

            if uid in last_seen:
                delta = current_time - last_seen[uid]
            else:
                delta = -1

            deltas.append(delta)

            last_seen[uid] = current_time

        X["uid_time_delta"] = deltas

        X.drop(columns=["uid"], inplace=True)

        return X

## Product Amount Interaction Adder (Not Used)

In [44]:
from sklearn.base import BaseEstimator, TransformerMixin


class ProductAmountInteractionAdder(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        X = X.copy()

        if "ProductCD" in X.columns and "TransactionAmt" in X.columns:

            X["ProductAmtInteraction"] = (
                X["ProductCD"].astype(str) + "_" +
                X["TransactionAmt"].round(0).astype(str)
            )

        return X

## Feature Engineering pipeline

In [49]:
feature_engineering_pipeline = Pipeline([
    ("uid_deviation", UIDAmountDeviationAdder())
])

# Feature Selection

## IVV (Not Used)

In [50]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class IVValidator(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.02, bins=10):
        self.threshold = threshold
        self.bins = bins
        self.cols_to_keep_ = None
        self.iv_scores_ = {}

    def _calculate_iv(self, x, y):
        temp = pd.DataFrame({"x": x, "y": y})
        temp["x"] = temp["x"].fillna("Missing")

        if pd.api.types.is_numeric_dtype(temp["x"]):
            try:
                temp["x"] = pd.qcut(temp["x"], q=self.bins, duplicates="drop")
            except Exception:
                temp["x"] = temp["x"].astype(str)
        else:
            temp["x"] = temp["x"].astype(str)

        grouped = temp.groupby("x")["y"].agg(["count", "sum"])
        grouped["non_event"] = grouped["count"] - grouped["sum"]

        event_total = grouped["sum"].sum()
        non_event_total = grouped["non_event"].sum()

        grouped["event_rate"] = grouped["sum"] / (event_total + 1e-6)
        grouped["non_event_rate"] = grouped["non_event"] / (non_event_total + 1e-6)

        grouped["woe"] = np.log(
            (grouped["event_rate"] + 1e-6) /
            (grouped["non_event_rate"] + 1e-6)
        )

        grouped["iv"] = (
            grouped["event_rate"] - grouped["non_event_rate"]
        ) * grouped["woe"]

        return grouped["iv"].sum()

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y)

        for col in X.columns:
            try:
                self.iv_scores_[col] = self._calculate_iv(X[col], y)
            except Exception:
                self.iv_scores_[col] = 0

        self.cols_to_keep_ = [
            col for col, iv in self.iv_scores_.items()
            if iv >= self.threshold
        ]

        return self

    def transform(self, X):
        X = X.copy()
        return X[self.cols_to_keep_]

## RFE

In [54]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import RFE
from xgboost import XGBClassifier


class XGBRFESelector(BaseEstimator, TransformerMixin):

    def __init__(self, n_features_to_select=300, step=0.4):
        self.n_features_to_select = n_features_to_select
        self.step = step
        self.selector_ = None

    def fit(self, X, y=None):

        estimator = XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )

        self.selector_ = RFE(
            estimator=estimator,
            n_features_to_select=self.n_features_to_select,
            step=self.step
        )

        self.selector_.fit(X, y)

        return self

    def transform(self, X):
        return self.selector_.transform(X)

In [55]:
feature_selection_pipeline = Pipeline([
    ("rfe_selector", XGBRFESelector(
        n_features_to_select=300,
        step=0.4
    ))
])

# XGBoost Pipeline

In [61]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("feature_engineering", feature_engineering_pipeline),
    ("cleaning", cleaning_pipeline),
    ("preprocessing", AutoPreprocessor()),
    ("feature_selection", feature_selection_pipeline),
    ("model", model)
])

# Log

In [62]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, log_loss

mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGB_woe"):

    threshold = 0.5

    pipeline.fit(X_train, y_train)

    train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
    val_pred_proba = pipeline.predict_proba(X_val)[:, 1]

    val_pred = (val_pred_proba >= threshold).astype(int)

    train_roc_auc = roc_auc_score(y_train, train_pred_proba)
    val_roc_auc = roc_auc_score(y_val, val_pred_proba)

    train_log_loss = log_loss(y_train, train_pred_proba)
    val_log_loss = log_loss(y_val, val_pred_proba)

    overfit_gap = train_roc_auc - val_roc_auc
    overfit_loss_gap = val_log_loss - train_log_loss

    precision = precision_score(y_val, val_pred)
    recall = recall_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred)

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("baseline_type", "raw_data_minimal_preprocessing")

    mlflow.log_param("numeric_imputation", "median")
    mlflow.log_param("categorical_imputation", "most_frequent")
    mlflow.log_param("encoding", "OHE+WOE")

    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("objective", "binary:logistic")
    mlflow.log_param("eval_metric", "logloss")

    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_jobs", -1)

    mlflow.log_param("feature_engineering", "True")
    mlflow.log_param("engineered_feature", "UID")
    mlflow.log_param("feature_selection", "True")
    mlflow.log_param("threshold", threshold)

    mlflow.log_metric("train_roc_auc", train_roc_auc)
    mlflow.log_metric("val_roc_auc", val_roc_auc)
    mlflow.log_metric("overfit_gap", overfit_gap)

    mlflow.log_metric("train_log_loss", train_log_loss)
    mlflow.log_metric("val_log_loss", val_log_loss)
    mlflow.log_metric("overfit_loss_gap", overfit_loss_gap)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="xgboost"
    )

    print("Train ROC-AUC:", train_roc_auc)
    print("Val ROC-AUC:", val_roc_auc)
    print("Overfit Gap:", overfit_gap)
    print("Train Log Loss:", train_log_loss)
    print("Val Log Loss:", val_log_loss)
    print("Overfit Loss Gap:", overfit_loss_gap)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)

2026/05/07 00:19:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 00:19:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Train ROC-AUC: 0.9572611464166065
Val ROC-AUC: 0.9427477723449207
Overfit Gap: 0.014513374071685847
Train Log Loss: 0.06071706491262233
Val Log Loss: 0.06817607663641703
Overfit Loss Gap: 0.007459011723794699
Precision: 0.9247361174850849
Recall: 0.4875393176869102
F1: 0.6384664131812421
🏃 View run XGB_woe at: https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4/runs/a18c6c695edd4d3790398f11d4d76e23
🧪 View experiment at: https://dagshub.com/izere23/Assignment-2-IEEE-CIS-Fraud-Detection.mlflow/#/experiments/4
